In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import SGD
# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

In [ ]:
# 2. Create TensorDataset objects




In [ ]:
# 3. Create DataLoaders
from torch.utils.data import DataLoader

# DataLoader for training data
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2
)
# DataLoader for test/validation data
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)




In [ ]:
# 4. Print shape of one batch
# Get the first batch from the training DataLoader
X_batch, y_batch = next(iter(train_loader))
print(f"Training batch input shape: {X_batch.shape}")
print(f"Training batch labels shape: {y_batch.shape}")



In [ ]:
# 5. Display sample images



In [ ]:
# Task 1: Write your model class here:
class NN3Layer(nn.Module):
  def __init__(self, input_dim, hidden_dim):
    super(NN3Layer, self).__init__()

    self.layer1 = nn.Linear(input_dim, hidden_dim)


    self.layer2 = nn.Linear(hidden_dim, hidden_dim)
    self.layer4 = nn.Linear(hidden_dim, hidden_dim)


    self.layer3 = nn.Linear(hidden_dim, 1)

    # TODO: Define ReLU activation
    self.relu = nn.ReLU()

  def forward(self, x):
    # TODO: First hidden layer with ReLU
    a1 = self.relu(self.layer1(x))

    # TODO: Second hidden layer with ReLU
    a2 = self.relu(self.layer2(a1))

    # TODO: Output layer (no activation for regression)
    output = self.layer3(a2)

    return output

In [ ]:
# Task 2: Write your training loop here:
def train_one_epoch(model, optimizer, criterion, train_loader, device):
  # Set the model to training mode
  model.train()

  running_loss = 0.0

  for X_batch, y_batch in train_loader:
    # Move batch to the selected device
    X_batch = X_batch.view(32,-1).to(device)
    y_batch = y_batch.view(-1,1).to(device)

    #  Forward pass - get model predictions
    outputs = model(X_batch)

    #  Compute loss using criterion
    loss = criterion(outputs, y_batch)

    #  Backward pass & optimization
    # Step 1: Clear previous gradients
    optimizer.zero_grad()


    # Step 2: Compute gradients (backward pass)
    loss.backward()


    # Step 3: Update model parameters
    optimizer.step()


    running_loss += loss.item()

  # Calculate average loss over all batches
  avg_loss = running_loss / len(train_loader)

  return avg_loss

In [ ]:
# Task 3: Write your validation loop here:
def validate(model, criterion, test_loader, device):
  #  Set the model to evaluation mode
  model.eval()

  running_loss = 0.0

  #  Disable gradient computation using torch.no_grad()
  with torch.no_grad():
    for X_batch, y_batch in test_loader:
      #  Move data to device
      X_batch = X_batch.view(32,-1).to(device)
      y_batch = y_batch.view(-1,1).to(device)

      # Forward pass - get model predictions
      outputs = model(X_batch)
      #  Compute loss using criterion
      loss = criterion(outputs, y_batch)

      running_loss += loss.item()

  avg_loss = running_loss / len(test_loader)

  return avg_loss

In [ ]:
# Task 4: Define device, model, loss, optimizer:
#  Set up the device (use GPU if available, otherwise CPU)
from torch.optim import AdamW
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Model parameters
input_dim = 784  # Number of tabular features
hidden_dim = 64                # Design choice (feel free to experiment!)

#  Instantiate the model and move it to the device
model = NN3Layer(input_dim, hidden_dim).to(device)

# Print the model architecture
print("Model Architecture:\n")
print(model)

# Calculate the total number of trainable parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal trainable parameters: {total_params}")


#  Define criterion (loss function) - use MSELoss for regression
criterion = nn.MSELoss()  # <Replace None with your code>

#  Define optimizer - use AdamW with the model parameters and learning rate
optimizer = AdamW(model.parameters(), lr=learning_rate) # <Replace None with your code>

In [ ]:
# Task 5: Start training for 20 epochs:
# Hyperparameters
num_epochs = 20
learning_rate = 0.001

# Run Training
train_losses = []
val_losses = []

print('Starting Training...')
for epoch in range(num_epochs):
  #  Train one epoch using the train_one_epoch function
  train_loss = train_one_epoch(model, optimizer, criterion, train_loader, device)

  #  Validate using the validate function
  val_loss = validate(model, criterion, test_loader, device)

  train_losses.append(train_loss)
  val_losses.append(val_loss)

  print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')

print('Training Complete!')

In [ ]:
# Task 1: Write your code here:

In [ ]:
# Task 2 (Bonus): Write your code here: